
# 05b_feature_construction_and_selection

**Goals**:
1. Build reduced clinically interpretable feature spaces.
2. Create two temporally coherent workflows:

EARLY PREDICTION (0–3 days)
- mean_0_3
- delta_0_3
- trajectory_direction_0_3

FULL TRAJECTORY CHARACTERIZATION (0–7 days)
- mean_0_7
- delta_0_7
- trajectory_direction_0_7

3. Reduce dimensionality and redundancy.
4. Remove near-constant variables.
5. Improve downstream temporal generalization.

**Outputs**:
- 05b_features_0_3.parquet
- 05b_features_0_7.parquet
- 05b_feature_dictionary.xlsx
- 05b_selected_feature_summary.xlsx

In [ ]:
# ============================================================
# 01. IMPORTS
# ============================================================

import sys
import numpy as np
import pandas as pd

from pathlib import Path

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

In [ ]:
# ============================================================
# 02. PROJECT PATHS
# ============================================================

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/"
    "pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

OUTPUT_05B = (
    PROJECT_CONSTANTS_DIR /
    "outputs" /
    "05b_feature_construction_and_selection"
)

OUTPUT_05B.mkdir(parents=True, exist_ok=True)

print("OUTPUT_03:", OUTPUT_03)
print("OUTPUT_05:", OUTPUT_05)
print("OUTPUT_05B:", OUTPUT_05B)

In [ ]:
# ============================================================
# 03. LOAD DATASETS
# ============================================================

df_model = pd.read_parquet(
    OUTPUT_03 / "03_model_features.parquet"
)

df_daily = pd.read_parquet(
    OUTPUT_03 / "03_daily_labs.parquet"
)

# Clinical dataset from notebook 01

CLINICAL_PATH = (
    PROJECT_CONSTANTS_DIR /
    "outputs" /
    "01_data_preparation" /
    "01_clinical.parquet"
)

df_clinical = pd.read_parquet(CLINICAL_PATH)

print("df_clinical:", df_clinical.shape)

print("Model dataset:", df_model.shape)
print("Daily labs:", df_daily.shape)

display(df_model.head())
display(df_daily.head())

In [ ]:
# ============================================================
# MERGE ADDITIONAL CLINICAL VARIABLES
# ============================================================

clinical_extra_cols = [
    "episode_key",
    "admBarthelScore",
    "admBradenScore",
    "admECOG",
]

# Keep only available columns
clinical_extra_cols = [
    c for c in clinical_extra_cols
    if c in df_clinical.columns
]

print("Clinical variables found:")
print(clinical_extra_cols)

df_model = df_model.merge(
    df_clinical[clinical_extra_cols],
    on="episode_key",
    how="left",
)

print("\ndf_model after merge:", df_model.shape)

display(
    df_model[
        [
            c for c in [
                "admBarthelScore",
                "admBradenScore",
                "admECOG",
            ]
            if c in df_model.columns
        ]
    ].head()
)

In [ ]:
# ============================================================
# 04. DEFINE OUTCOMES
# ============================================================

OUTCOME_COLS = [
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
]

display(df_model[OUTCOME_COLS].mean())

In [ ]:
# ============================================================
# 05. CORE BIOMARKERS
# ============================================================

try:
    CORE_BIOMARKERS
except NameError:
    CORE_BIOMARKERS = [
        "CRP",
        "WBC",
        "Neutrophils",
        "Lymphocytes",
        "Hemoglobin",
        "Platelets",
        "Creatinine",
        "Urea",
        "Sodium",
        "Potassium",
    ]

CORE_BIOMARKERS = [
    b for b in CORE_BIOMARKERS
    if b in df_daily["biomarker"].unique()
]

print("Core biomarkers:")
print(CORE_BIOMARKERS)

# CLINICAL RATIONALE

This pipeline intentionally prioritizes clinically interpretable
and temporally coherent features.

For each biomarker, only three longitudinal summaries are retained:

1. mean value within the observation window
2. delta within the observation window
3. trajectory direction within the observation window

This design aims to:
- reduce dimensionality,
- reduce overfitting,
- improve interpretability,
- improve temporal generalization,
- preserve clinically meaningful trajectory information.


In [ ]:
# ============================================================
# 07. REDUCED CLINICAL FEATURE SET
# ============================================================

clinical_core_candidates = [
    "age",
    "sex",
    "bmi",
    "adm_mews",
    "admBarthelScore",
    "admBradenScore",
    "admECOG",
    "ccsi",
    "solidTumor_cat",
]

clinical_core_features = [
    c for c in clinical_core_candidates
    if c in df_model.columns
]

clinical_survival_extra_features = [
    c for c in ["LOS_days"]
    if c in df_model.columns
]

print("Clinical core features:")
print(clinical_core_features)

print("\nN clinical features:", len(clinical_core_features))

print("\nSurvival-only extra features:")
print(clinical_survival_extra_features)

In [ ]:
# ============================================================
# 08. PREPARE LABS
# ============================================================

df_labs = df_daily.copy()

df_labs = df_labs[
    df_labs["biomarker"].isin(CORE_BIOMARKERS)
].copy()

df_labs["lab_day"] = pd.to_numeric(
    df_labs["lab_day"],
    errors="coerce"
)

df_labs["value"] = pd.to_numeric(
    df_labs["value"],
    errors="coerce"
)

df_labs = df_labs.dropna(
    subset=["episode_key", "biomarker", "lab_day", "value"]
)

df_labs = df_labs[
    df_labs["lab_day"].between(0, 7)
].copy()

print(df_labs.shape)
display(df_labs.head())

In [ ]:
# ============================================================
# 09. HELPER FUNCTION
# ============================================================

def summarize_window(
    labs_df,
    start_day,
    end_day,
    suffix
):
    """
    Creates:
    - mean
    - delta
    - trajectory_direction
    """

    rows = []

    for episode_key, ep_data in labs_df.groupby("episode_key"):

        row = {
            "episode_key": episode_key
        }

        for biomarker in CORE_BIOMARKERS:

            temp = ep_data[
                (ep_data["biomarker"] == biomarker)
                & (
                    ep_data["lab_day"].between(
                        start_day,
                        end_day
                    )
                )
            ].copy()

            if temp.empty:

                row[f"{biomarker}_mean_{suffix}"] = np.nan
                row[f"{biomarker}_delta_{suffix}"] = np.nan
                row[f"{biomarker}_trajectory_direction_{suffix}"] = (
                    "not_observable"
                )

                continue

            temp = temp.sort_values("lab_day")

            daily = (
                temp
                .groupby("lab_day", as_index=False)["value"]
                .mean()
                .sort_values("lab_day")
            )

            first_value = daily["value"].iloc[0]
            last_value = daily["value"].iloc[-1]

            mean_value = daily["value"].mean()

            delta = last_value - first_value

            row[f"{biomarker}_mean_{suffix}"] = mean_value

            row[f"{biomarker}_delta_{suffix}"] = delta

            # trajectory direction
            if np.abs(delta) < 1e-6:
                direction = "stable"

            elif delta > 0:
                direction = "increasing"

            else:
                direction = "decreasing"

            row[
                f"{biomarker}_trajectory_direction_{suffix}"
            ] = direction

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 10. BUILD 0–3 SUMMARIES
# ============================================================

labs_0_3 = summarize_window(
    labs_df=df_labs,
    start_day=0,
    end_day=3,
    suffix="0_3"
)

print(labs_0_3.shape)

display(labs_0_3.head())

In [ ]:
# ============================================================
# 11. BUILD 0–7 SUMMARIES
# ============================================================

labs_0_7 = summarize_window(
    labs_df=df_labs,
    start_day=0,
    end_day=7,
    suffix="0_7"
)

print(labs_0_7.shape)

display(labs_0_7.head())

In [ ]:
# ============================================================
# DEBUG LONGITUDINAL FEATURES
# ============================================================

all_longitudinal_cols = [
    c for c in (
        list(labs_0_3.columns)
        + list(labs_0_7.columns)
    )
    if (
        "_mean_" in c
        or "_delta_" in c
        or "_trajectory_direction_" in c
    )
]

print("Total longitudinal features:", len(all_longitudinal_cols))

display(sorted(all_longitudinal_cols))

print("\nDelta features:")
display([
    c for c in all_longitudinal_cols
    if "_delta_" in c
])

In [ ]:
# ============================================================
# 12. MERGE FEATURES — CLEAN VERSION
# ============================================================

df_features = df_model.copy()

# Remove any pre-existing longitudinal columns from df_model
# to avoid _x / _y suffixes after merge.
preexisting_longitudinal_cols = [
    c for c in df_features.columns
    if (
        "_mean_0_3" in c
        or "_delta_0_3" in c
        or "_trajectory_direction_0_3" in c
        or "_mean_0_7" in c
        or "_delta_0_7" in c
        or "_trajectory_direction_0_7" in c
    )
]

print("Removing pre-existing longitudinal columns from df_model:")
print(len(preexisting_longitudinal_cols))
display(preexisting_longitudinal_cols)

df_features = df_features.drop(
    columns=preexisting_longitudinal_cols,
    errors="ignore"
)

df_features = df_features.merge(
    labs_0_3,
    on="episode_key",
    how="left",
    validate="one_to_one"
)

df_features = df_features.merge(
    labs_0_7,
    on="episode_key",
    how="left",
    validate="one_to_one"
)

print("df_features shape:", df_features.shape)

# Check final longitudinal columns
longitudinal_cols = [
    c for c in df_features.columns
    if (
        "_mean_0_3" in c
        or "_delta_0_3" in c
        or "_trajectory_direction_0_3" in c
        or "_mean_0_7" in c
        or "_delta_0_7" in c
        or "_trajectory_direction_0_7" in c
    )
]

print("Final longitudinal columns:", len(longitudinal_cols))
display(sorted(longitudinal_cols))

In [ ]:
# ============================================================
# DEBUG MERGED 0–3 COLUMN NAMES
# ============================================================

for biomarker in CORE_BIOMARKERS:
    print("\n", biomarker)
    display([
        c for c in df_features.columns
        if biomarker in c and "0_3" in c
    ])

In [ ]:
# ============================================================
# 13. METADATA COLUMNS
# ============================================================

metadata_cols = [
    "episode_key",
    "pubID",
    "temporal_cohort",
    "metastatic_group",
    "hospitalWard",
    "endDate",
    "dateOfDeath",
]

metadata_cols = [
    c for c in metadata_cols
    if c in df_features.columns
]

print(metadata_cols)

In [ ]:
# ============================================================
# 14. EARLY FEATURE SET (0–3) — FORCED FROM labs_0_3
# ============================================================

early_lab_features = []

for biomarker in CORE_BIOMARKERS:
    early_lab_features.extend([
        f"{biomarker}_mean_0_3",
        f"{biomarker}_delta_0_3",
        f"{biomarker}_trajectory_direction_0_3",
    ])

missing_early = [
    c for c in early_lab_features
    if c not in df_features.columns
]

print("Requested early lab features:", len(early_lab_features))
print("Missing early lab features:", len(missing_early))
display(missing_early)

assert len(missing_early) == 0, "Some requested 0–3 lab features are missing from df_features."

features_0_3 = (
    metadata_cols
    + OUTCOME_COLS
    + clinical_core_features
    + early_lab_features
)

features_0_3 = list(dict.fromkeys(features_0_3))

df_0_3 = df_features.loc[:, features_0_3].copy()

print("df_0_3 shape:", df_0_3.shape)
display(df_0_3.head())

In [ ]:
print("Early lab features:")
display(early_lab_features)

print("\nCounts:")
print("mean_0_3:", len([c for c in early_lab_features if "_mean_0_3" in c]))
print("delta_0_3:", len([c for c in early_lab_features if "_delta_0_3" in c]))
print("trajectory_0_3:", len([c for c in early_lab_features if "_trajectory_direction_0_3" in c]))

print("\nColumns in df_0_3:")
display([c for c in df_0_3.columns if "_0_3" in c])

In [ ]:
# ============================================================
# 15. FULL FEATURE SET (0–7)
# ============================================================

full_lab_features = []

for biomarker in CORE_BIOMARKERS:

    candidate_features = [
        f"{biomarker}_mean_0_7",
        f"{biomarker}_delta_0_7",
        f"{biomarker}_trajectory_direction_0_7",
    ]

    existing_features = [
        c for c in candidate_features
        if c in df_features.columns
    ]

    full_lab_features.extend(existing_features)

print("Full longitudinal features:", len(full_lab_features))
display(full_lab_features)

features_0_7 = (
    metadata_cols
    + OUTCOME_COLS
    + clinical_core_features
    + clinical_survival_extra_features
    + full_lab_features
)

features_0_7 = list(dict.fromkeys(features_0_7))

df_0_7 = df_features.loc[:, features_0_7].copy()

print("\ndf_0_7 shape:", df_0_7.shape)

display(df_0_7.head())

In [ ]:
# ============================================================
# 16. NEAR-CONSTANT FEATURE HELPER
# ============================================================

def identify_near_constant_features(
    df,
    protected_cols,
    dominance_threshold=0.985
):

    rows = []

    candidate_cols = [
        c for c in df.columns
        if c not in protected_cols
    ]

    for col in candidate_cols:

        s = df[col].dropna()

        if s.empty:
            continue

        n_unique = s.nunique()

        # Numerical variables
        if pd.api.types.is_numeric_dtype(s):

            if n_unique <= 1:

                rows.append({
                    "feature": col,
                    "reason": "numeric_constant",
                    "dominance": np.nan,
                    "n_unique": n_unique,
                })

        # Categorical variables
        else:

            value_counts = s.value_counts(normalize=True)

            dominance = value_counts.iloc[0]

            if dominance >= dominance_threshold:

                rows.append({
                    "feature": col,
                    "reason": "dominant_category",
                    "dominance": dominance,
                    "n_unique": n_unique,
                })

    # IMPORTANT:
    # ensure columns always exist even if rows is empty
    return pd.DataFrame(
        rows,
        columns=[
            "feature",
            "reason",
            "dominance",
            "n_unique",
        ]
    )

In [ ]:
# ============================================================
# 17. REMOVE NEAR-CONSTANT VARIABLES
# ============================================================

protected_longitudinal_patterns = [
    "_mean_",
    "_delta_",
    "_trajectory_direction_",
]

protected_cols = list(dict.fromkeys(
    metadata_cols
    + OUTCOME_COLS
    + clinical_core_features
))

near_constant_0_3 = identify_near_constant_features(
    df=df_0_3,
    protected_cols=protected_cols,
)

near_constant_0_7 = identify_near_constant_features(
    df=df_0_7,
    protected_cols=protected_cols,
)

print("Near-constant features 0–3:")
display(near_constant_0_3)

print("Near-constant features 0–7:")
display(near_constant_0_7)

remove_0_3 = [
    f for f in (
        near_constant_0_3["feature"].tolist()
        if not near_constant_0_3.empty
        else []
    )
    if not any(
        p in f
        for p in protected_longitudinal_patterns
    )
]

remove_0_7 = [
    f for f in (
        near_constant_0_7["feature"].tolist()
        if not near_constant_0_7.empty
        else []
    )
    if not any(
        p in f
        for p in protected_longitudinal_patterns
    )
]

df_0_3_filtered = df_0_3.drop(
    columns=remove_0_3,
    errors="ignore"
)

df_0_7_filtered = df_0_7.drop(
    columns=remove_0_7,
    errors="ignore"
)

print("\n0–3:")
print(df_0_3.shape, "->", df_0_3_filtered.shape)

print("\n0–7:")
print(df_0_7.shape, "->", df_0_7_filtered.shape)

print("\nRemoved 0–3:", len(remove_0_3))
print("Removed 0–7:", len(remove_0_7))

In [ ]:
clinical_check = [
    "admBarthelScore",
    "admBradenScore",
    "admECOG",
]

print("In df_model:")
print([c for c in clinical_check if c in df_model.columns])

print("In df_features:")
print([c for c in clinical_check if c in df_features.columns])

print("In df_0_3 before filter:")
print([c for c in clinical_check if c in df_0_3.columns])

print("In df_0_3_filtered after filter:")
print([c for c in clinical_check if c in df_0_3_filtered.columns])

print("Removed near-constant 0–3:")
display(near_constant_0_3[near_constant_0_3["feature"].isin(clinical_check)])

print("Removed near-constant 0–7:")
display(near_constant_0_7[near_constant_0_7["feature"].isin(clinical_check)])

In [ ]:
# ============================================================
# 18. FEATURE DICTIONARY
# ============================================================

rows = []

for dataset_name, df_current in [
    ("features_0_3", df_0_3_filtered),
    ("features_0_7", df_0_7_filtered),
]:

    for col in df_current.columns:

        rows.append({
            "dataset": dataset_name,
            "feature": col,
            "dtype": str(df_current[col].dtype),
            "missing_rate": df_current[col].isna().mean(),
            "n_unique": df_current[col].nunique(dropna=True),
        })

feature_dictionary = pd.DataFrame(rows)

display(feature_dictionary.head())

In [ ]:
# ============================================================
# 19. FEATURE SUMMARY
# ============================================================

summary_rows = []

for name, df_current in [
    ("0_3", df_0_3_filtered),
    ("0_7", df_0_7_filtered),
]:

    model_features = [
        c for c in df_current.columns
        if c not in (
            metadata_cols
            + OUTCOME_COLS
        )
    ]

    summary_rows.append({
        "dataset": name,
        "n_rows": df_current.shape[0],
        "n_total_columns": df_current.shape[1],
        "n_model_features": len(model_features),
    })

feature_summary = pd.DataFrame(summary_rows)

display(feature_summary)

In [ ]:
# ============================================================
# 20. FINAL QC
# ============================================================

for name, df_current in [
    ("0_3", df_0_3_filtered),
    ("0_7", df_0_7_filtered),
]:

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print(df_current.shape)

    print("\nMissingness:")
    display(
        df_current.isna()
        .mean()
        .sort_values(ascending=False)
        .head(20)
    )

In [ ]:
# ============================================================
# 21. SAVE OUTPUTS
# ============================================================

path_0_3 = (
    OUTPUT_05B /
    "05b_features_0_3.parquet"
)

path_0_7 = (
    OUTPUT_05B /
    "05b_features_0_7.parquet"
)

dictionary_path = (
    OUTPUT_05B /
    "05b_feature_dictionary.xlsx"
)

summary_path = (
    OUTPUT_05B /
    "05b_selected_feature_summary.xlsx"
)

df_0_3_filtered.to_parquet(
    path_0_3,
    index=False
)

df_0_7_filtered.to_parquet(
    path_0_7,
    index=False
)

with pd.ExcelWriter(dictionary_path) as writer:

    feature_dictionary.to_excel(
        writer,
        sheet_name="feature_dictionary",
        index=False
    )

with pd.ExcelWriter(summary_path) as writer:

    feature_summary.to_excel(
        writer,
        sheet_name="feature_summary",
        index=False
    )

print("Saved:")
print(path_0_3)
print(path_0_7)
print(dictionary_path)
print(summary_path)

In [ ]:
# ============================================================
# 22. FINAL NOTE
# ============================================================

print("""
Feature construction completed.

Two modelling datasets were exported:

1. 05b_features_0_3.parquet
   Early hospitalization prediction workflow.

2. 05b_features_0_7.parquet
   Full hospitalization trajectory characterization workflow.

Both datasets intentionally prioritize:
- reduced dimensionality,
- temporal coherence,
- clinical interpretability,
- trajectory-based summaries,
- improved generalization potential.
""")